# Rețele de calculatoare · Săptămâna 3 — Notebook de aplicații (Google Colab)
### Stratul aplicație: socket-uri, Web-ul și HTTP/1.1

Notebook însoțitor al **pachetului didactic din Săptămâna 3** (Cursul 3, Seminarul 3, Laboratorul 3 Linux, Laboratorul 3 Windows).

| Partea | În pachetul didactic | În acest notebook |
|---|---|---|
| **A** | Cursul 3 | Client–server vs. P2P în cifre; socket-uri, porturi și 4-tupluri observate în direct; granițele mesajelor la TCP vs. UDP; anatomia unui URL; un calculator al timpului de încărcare a paginii; un mic **server și client HTTP/1.1 scrise direct pe socket-uri**, care implementează Content-Length, corpuri chunked, conexiuni persistente și GET condițional; ordinea octeților în rețea |
| **B** | Seminarul 3 | Problemele 1–10 calculate și *rulate*: un releu care taie un flux TCP în locuri aleatoare, coduri de stare produse de un server real, 4-tupluri în direct și protocolul de upload de fișiere din Problema 10 implementat complet |
| **C** | Laboratorul 3 (Linux) | Toate programele C din fișă (`netio.h`, `echo_client.c`, `echo_server.c`, `upper_server.c`, `blast.c`, `msg_client.c`, `msg_server.c`, plus `file_client.c` și `broken_pipe.c`) scrise, compilate cu `make` și rulate pe VM-ul Colab, Sarcinile 1–8 și versiunile Python |
| **D** | Laboratorul 3 (Windows) | Programele Winsock **compilate încrucișat pentru Windows** cu MinGW-w64 (descărcați fișierele `.exe` și rulați-le pe PC), diferențele Winsock/POSIX, codurile de eroare Winsock și, opțional, partea PowerShell din fișă |

**Cum se folosește.** Rulați celulele în ordine (merge și `Runtime → Run all`). Serverele sunt pornite în fundal de notebook, deci nu aveți nevoie niciodată de un al doilea terminal. Celulele marcate cu ✍️ sunt pentru răspunsurile voastre la Q1–Q16, astfel încât notebook-ul să fie raportul vostru (`File → Download → .ipynb`).

> Fișierele sursă ale laboratorului (C și Python) sunt păstrate exact ca în pachetul didactic, cu comentariile în engleză.

## 0 · Pregătire

In [ ]:
# ---------------------------------------------------------------
#  Funcții ajutătoare folosite în tot notebook-ul. Rulați întâi această celulă.
# ---------------------------------------------------------------
import os, re, sys, time, math, json, socket, shutil, subprocess, threading, ipaddress
from collections import deque, Counter

IN_COLAB = 'google.colab' in sys.modules
SUDO = '' if os.geteuid() == 0 else 'sudo '

def sh(cmd, timeout=60, show=True):
    """Rulează o comandă shell, afișează ieșirea (stdout+stderr), nu ridică niciodată excepții."""
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        out = (r.stdout or '') + (r.stderr or '')
    except subprocess.TimeoutExpired as e:
        o = e.stdout or ''
        out = (o.decode(errors='replace') if isinstance(o, bytes) else o) + f'\n[oprit după {timeout} s]'
    if show:
        print(f'$ {cmd}\n{out.rstrip()}\n')
    return out

def have(tool):
    return shutil.which(tool) is not None

def default_route():
    """(interfața, gateway-ul) folosite pentru a ajunge în Internet."""
    out = sh('ip route get 1.1.1.1', show=False)
    dev = re.search(r'\bdev (\S+)', out)
    via = re.search(r'\bvia (\S+)', out)
    src = re.search(r'\bsrc (\S+)', out)
    return (dev.group(1) if dev else 'any', via.group(1) if via else None, src.group(1) if src else None)

def resolve(host):
    try:
        return socket.gethostbyname(host)
    except Exception as e:
        print(f'nu pot rezolva {host}: {e}')
        return None

def tcp_rtt(host, port=443, n=5, timeout=3):
    """Măsoară RTT ca durată a unui connect() TCP (un schimb SYN / SYN-ACK).
    Funcționează și acolo unde ping-ul ICMP este blocat (ca pe multe VM-uri din cloud)."""
    ip = resolve(host)
    if not ip:
        return []
    rtts = []
    for _ in range(n):
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(timeout)
        t0 = time.perf_counter()
        try:
            s.connect((ip, port))
            rtts.append((time.perf_counter() - t0) * 1000)
        except Exception as e:
            print(f'  conectarea la {ip}:{port} a eșuat: {e}')
        finally:
            s.close()
        time.sleep(0.2)
    if rtts:
        print(f'RTT prin conectare TCP la {host} ({ip}:{port}): min/med/max = '
              f'{min(rtts):.2f}/{sum(rtts)/len(rtts):.2f}/{max(rtts):.2f} ms  ({len(rtts)}/{n} răspunsuri)')
    return rtts

def fmt_time(seconds):
    """Afișează frumos o durată."""
    s = abs(seconds)
    if s < 1e-6: return f'{seconds*1e9:.3g} ns'
    if s < 1e-3: return f'{seconds*1e6:.4g} µs'
    if s < 1:    return f'{seconds*1e3:.4g} ms'
    if s < 120:  return f'{seconds:.4g} s'
    if s < 7200: return f'{seconds/60:.4g} min'
    if s < 2*86400: return f'{seconds/3600:.4g} h'
    return f'{seconds/86400:.4g} zile'

def check(name, got, expected, rel=0.02):
    ok = math.isclose(got, expected, rel_tol=rel, abs_tol=1e-12)
    print(f"{'✅' if ok else '❌'} {name}: calculat {got:.6g}, așteptat ≈ {expected:.6g}")
    return ok

print('Funcțiile ajutătoare au fost încărcate. Rulează în Colab:', IN_COLAB, '| root:', os.geteuid() == 0)

In [ ]:
# Compilator, compilator încrucișat pentru Windows, netcat. Circa 1–2 minute în Colab.
sh(f'{SUDO}apt-get -qq update > /dev/null 2>&1; DEBIAN_FRONTEND=noninteractive {SUDO}apt-get -qq install -y '
   'build-essential mingw-w64 netcat-openbsd iproute2 curl > /dev/null 2>&1; echo apt done', timeout=900)
for t in ['gcc', 'make', 'x86_64-w64-mingw32-gcc', 'nc', 'ss', 'curl']:
    print(f'{t:24s}', '✅' if have(t) else '❌ lipsește')

In [ ]:
import struct, random, signal, select, urllib.parse, email.utils, datetime
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 4.5), 'axes.grid': True, 'grid.alpha': .3})

LAB = os.path.abspath('lab3')
for d in ('lab3', 'lab3/python', 'lab3/win'):
    os.makedirs(d, exist_ok=True)

class Bg:
    """Un program care rulează în fundal (ca un al doilea terminal), cu ieșirea colectată."""
    ALL = []
    def __init__(self, cmd, cwd=LAB, settle=0.5):
        self.cmd = cmd
        self.p = subprocess.Popen('exec ' + cmd, shell=True, cwd=cwd, stdin=subprocess.PIPE,
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True)
        self.buf, self.mark = bytearray(), 0
        threading.Thread(target=self._reader, daemon=True).start()
        Bg.ALL.append(self)
        time.sleep(settle)
    def _reader(self):
        fd = self.p.stdout.fileno()
        while True:
            try:
                chunk = os.read(fd, 4096)
            except OSError:
                break
            if not chunk:
                break
            self.buf += chunk
    @property
    def pid(self):
        return self.p.pid
    def write(self, text):                     # tastăm în program
        self.p.stdin.write(text.encode()); self.p.stdin.flush()
    def eof(self):                             # ca apăsarea Ctrl+D
        self.p.stdin.close()
    def output(self):
        return self.buf.decode(errors='replace')
    def new_output(self):                      # ieșirea de la ultimul apel
        out = self.buf[self.mark:].decode(errors='replace'); self.mark = len(self.buf); return out
    def running(self):
        return self.p.poll() is None
    def stop(self, sig=signal.SIGTERM):        # ca apăsarea Ctrl+C (SIGINT) sau kill (SIGTERM)
        if self.running():
            try:
                os.killpg(self.p.pid, sig); self.p.wait(timeout=3)
            except Exception:
                try: os.killpg(self.p.pid, signal.SIGKILL)
                except Exception: pass
        return self.p.returncode
    @classmethod
    def stop_all(cls):
        for b in cls.ALL: b.stop()
        cls.ALL.clear()

def show(title, text):
    print(f'── {title} ' + '─' * max(0, 60 - len(title)))
    print(text.rstrip() if text.strip() else '(nimic)')

---
# Partea A · Cursul 3 — cursul în cod

## A.1 Arhitecturi de aplicații: client–server vs. peer-to-peer
Cât durează să dăm un fișier de dimensiune **F** la **N** gazde? Cu un server cu rata de upload *uₛ* și clienți cu rata de download *d*:
client–server `D = max(N·F/uₛ, F/d)`; peer-to-peer, unde fiecare peer încarcă și el cu *u*: `D = max(F/uₛ, F/d, N·F/(uₛ + N·u))`. P2P este **auto-scalabil**: fiecare peer nou aduce cerere *și* capacitate.

In [ ]:
F, us, u, d = 1e9, 30e6, 2e6, 20e6        # fișier de 1 Gbit, server 30 Mbit/s upload, peer-i 2 Mbit/s upload, 20 Mbit/s download
N = np.arange(1, 1001)
cs = np.maximum(N * F / us, F / d)
p2p = np.maximum.reduce([np.full(N.shape, F / us), np.full(N.shape, F / d), N * F / (us + N * u)])
plt.plot(N, cs / 3600, label='client–server'); plt.plot(N, p2p / 3600, label='peer-to-peer')
plt.xlabel('numărul de gazde N'); plt.ylabel('timpul de distribuire a fișierului (ore)'); plt.legend()
plt.title('Client–server crește liniar; P2P se aplatizează'); plt.show()
for n in (10, 100, 1000):
    print(f'N = {n:4d}: client–server {cs[n-1]/60:7.1f} min   P2P {p2p[n-1]/60:6.1f} min')

## A.2 Procese, porturi și 4-tuplul — observate în direct
Un proces server ascultă pe un port; fiecare client primește un port **efemer** de la sistemul său de operare; serverul deosebește conexiunile după 4-tuplul *(IP sursă, port sursă, IP destinație, port destinație)*.

In [ ]:
lo, hi = open('/proc/sys/net/ipv4/ip_local_port_range').read().split()
print(f'Porturile efemere pe această gazdă Linux: {lo}–{hi}\n')

ls_ = socket.socket(); ls_.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
ls_.bind(('127.0.0.1', 0)); ls_.listen(8); sport = ls_.getsockname()[1]
clients = [socket.create_connection(('127.0.0.1', sport)) for _ in range(3)]
conns = [ls_.accept()[0] for _ in range(3)]
print(f'Serverul ascultă pe 127.0.0.1:{sport}; trei conexiuni, așa cum le vede serverul:')
for c in conns:
    (rip, rport), (lip, lport) = c.getpeername(), c.getsockname()
    print(f'   4-tuplu: ({rip}, {rport}, {lip}, {lport})')
sh(f'ss -tn | grep -E "State|:{sport}"')
for s in clients + conns + [ls_]: s.close()

## A.3 De ce au nevoie aplicațiile și ce oferă TCP și UDP
Diferența decisivă pentru programatori: **UDP păstrează granițele mesajelor, TCP nu.** Două apeluri `send()` de câte 5 octeți:

In [ ]:
# UDP: un send = o datagramă = un recvfrom
r = socket.socket(socket.AF_INET, socket.SOCK_DGRAM); r.bind(('127.0.0.1', 0))
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
for m in (b'HELLO', b'WORLD'): s.sendto(m, r.getsockname())
time.sleep(0.2)
print('Receptorul UDP a primit:', [r.recvfrom(100)[0] for _ in range(2)])
r.close(); s.close()

# TCP: un flux de octeți; receptorul primește tot ce a sosit
ls_ = socket.socket(); ls_.bind(('127.0.0.1', 0)); ls_.listen(1)
c = socket.create_connection(ls_.getsockname()); a, _ = ls_.accept()
c.send(b'HELLO'); c.send(b'WORLD')
time.sleep(0.2)
print('Receptorul TCP a primit:', a.recv(100), '  ← un singur recv, ambele mesaje lipite')
for x in (c, a, ls_): x.close()

In [ ]:
NEEDS = {  # aplicația: (pierderi de date, debit, sensibilă la timp)
    'Transfer de fișiere': ('fără pierderi', 'elastic', 'nu'),
    'E-mail': ('fără pierderi', 'elastic', 'nu'),
    'Pagini web': ('fără pierderi', 'elastic', 'oarecum (secunde)'),
    'Apeluri voce și video': ('tolerantă', 'audio kbit/s; video Mbit/s', 'da, ~100 ms'),
    'Video stocat (streaming)': ('tolerantă', 'Mbit/s', 'da, câteva secunde'),
    'Jocuri interactive': ('tolerantă', 'kbit/s și peste', 'da, zeci de ms'),
    'Mesagerie text': ('fără pierderi', 'elastic', 'da și nu'),
}
def suggest_transport(loss, timing):
    if loss == 'fără pierderi': return 'TCP (+ TLS dacă trebuie să fie sigură)'
    return 'UDP (+ mecanisme de tip RTP/QUIC deasupra)' if timing.startswith('da') else 'TCP sau UDP'
print(f'{"aplicația":24s} {"pierderi":14s} {"debit":27s} {"timp":20s} → transport')
for app, (loss, thr, tim) in NEEDS.items():
    print(f'{app:24s} {loss:14s} {thr:27s} {tim:20s} → {suggest_transport(loss, tim)}')

## A.4 Obiecte și URL-uri
`https://www.example.org:8443/courses/networks/week3.html?lang=en#exercises` — **fragmentul** este folosit doar de browser și nu este trimis niciodată serverului.

In [ ]:
url = 'https://www.example.org:8443/courses/networks/week3.html?lang=en#exercises'
u = urllib.parse.urlsplit(url)
for part in ('scheme', 'hostname', 'port', 'path', 'query', 'fragment'):
    print(f'{part:9s} {getattr(u, part)}')
print('portul implicit pentru http:', urllib.parse.urlsplit('http://example.org/').port or 80)
target = u.path + ('?' + u.query if u.query else '')
print('\nCe trimite de fapt browserul:')
print(f'GET {target} HTTP/1.1\r\nHost: {u.hostname}:{u.port}\r\n')

## A.5 Conexiunile HTTP și costul lor în timpi dus-întors
* **nepersistente**: `2 RTT + transmisie` pentru fiecare obiect (handshake + cerere);
* **persistente**: un singur handshake, apoi `1 RTT + transmisie` pentru fiecare obiect;
* **pipelining / HTTP/2**: toate cererile odată, un RTT, apoi obiectele unul după altul;
* **paralele**: mai multe conexiuni nepersistente simultan (transmisiile lor pot împărți o singură legătură).

In [ ]:
def page_load(n_objects, rtt, t_base=0.0, t_obj=0.0, mode='nonpersistent', parallel=1, shared_link=False):
    """Timpul de aducere a unui fișier de bază plus n_objects obiecte referite, de pe un singur server."""
    base = 2 * rtt + t_base                                   # prima conexiune: handshake + cerere
    if mode == 'nonpersistent':
        if parallel <= 1:
            return base + n_objects * (2 * rtt + t_obj)
        rounds, left, t = 0, n_objects, base
        while left > 0:
            k = min(parallel, left)
            t += 2 * rtt + (k * t_obj if shared_link else t_obj)
            left -= k
        return t
    if mode == 'persistent':
        return base + n_objects * (rtt + t_obj)
    if mode == 'pipelined':
        return base + (rtt + n_objects * t_obj if n_objects else 0)
    raise ValueError(mode)

RTT = 0.050
print('Exemplul rezolvat — fișier de bază + 10 imagini mici, RTT 50 ms, transmisia ignorată')
check('nepersistent, câte unul (ms)', page_load(10, RTT) * 1e3, 1100)
check('nepersistent, 5 în paralel (ms)', page_load(10, RTT, parallel=5) * 1e3, 300)
check('persistent (ms)', page_load(10, RTT, mode='persistent') * 1e3, 600)
check('persistent + pipelining / HTTP/2 (ms)', page_load(10, RTT, mode='pipelined') * 1e3, 150)

In [ ]:
# Figura 1 din curs: pagina de bază + 2 imagini, în RTT-uri
def timeline(ax, title, steps):
    y = 0
    for label, start, length, color in steps:
        ax.barh(y, length, left=start, color=color, edgecolor='k'); ax.text(start + length / 2, y, label, ha='center', va='center', fontsize=8)
        y += 1
    ax.set_yticks([]); ax.set_xlim(0, 6.2); ax.set_title(title, fontsize=10); ax.invert_yaxis(); ax.set_xlabel('timp (RTT)')

HS, RQ = '#ffd9a8', '#b8d8ff'
fig, axs = plt.subplots(1, 3, figsize=(13, 3.2))
timeline(axs[0], 'Nepersistent: 6 RTT', [('SYN', 0, 1, HS), ('GET bază', 1, 1, RQ), ('SYN', 2, 1, HS), ('GET img1', 3, 1, RQ), ('SYN', 4, 1, HS), ('GET img2', 5, 1, RQ)])
timeline(axs[1], 'Persistent: 4 RTT', [('SYN', 0, 1, HS), ('GET bază', 1, 1, RQ), ('GET img1', 2, 1, RQ), ('GET img2', 3, 1, RQ)])
timeline(axs[2], 'Pipelining / HTTP/2: 3 RTT', [('SYN', 0, 1, HS), ('GET bază', 1, 1, RQ), ('GET img1 + img2', 2, 1, RQ)])
plt.tight_layout(); plt.show()

objs = np.arange(0, 61)
for mode, kw in [('nepersistent', dict(mode='nonpersistent')), ('nepersistent ×6 în paralel', dict(mode='nonpersistent', parallel=6)),
                 ('persistent', dict(mode='persistent')), ('pipelining / HTTP/2', dict(mode='pipelined'))]:
    plt.plot(objs, [page_load(n, RTT, **kw) * 1e3 for n in objs], label=mode)
plt.xlabel('obiecte referite de pagină'); plt.ylabel('timpul de încărcare (ms), RTT = 50 ms'); plt.legend()
plt.title('Cu multe obiecte mici, domină timpii dus-întors'); plt.show()

## A.6 Mesajele HTTP/1.1 — un server și un client scrise direct pe socket-uri
Ca să vedem protocolul și nu o bibliotecă, mai jos este un **mic server HTTP/1.1 scris direct pe socket-uri** (circa 100 de linii). El implementează: conexiuni persistente (și `Connection: close`), `Content-Length`, `Transfer-Encoding: chunked`, GET condițional (`If-Modified-Since` → `304`), redirecționare, erori de autentificare, `405` și `400`, precum și `POST` cu cheie de idempotență. Numără conexiunile TCP și cererile, ca să putem *vedea* persistența.

In [ ]:
LAST_MOD = 'Fri, 16 Oct 2026 14:12:30 GMT'
LAST_MOD_TS = email.utils.parsedate_to_datetime(LAST_MOD)

class MiniHTTPServer(threading.Thread):
    def __init__(self, port=0):
        super().__init__(daemon=True)
        self.ls = socket.socket(); self.ls.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        self.ls.bind(('127.0.0.1', port)); self.ls.listen(16)
        self.port = self.ls.getsockname()[1]
        self.connections = self.requests = 0
        self.orders, self.seen_keys = [], {}
        self.start()

    def run(self):
        while True:
            try:
                c, _ = self.ls.accept()
            except OSError:
                return
            self.connections += 1
            threading.Thread(target=self.serve, args=(c,), daemon=True).start()

    def serve(self, c):
        rf = c.makefile('rb')
        with c:
            while True:
                line = rf.readline()
                if not line:
                    return                                          # clientul a închis
                parts = line.decode('latin-1').rstrip('\r\n').split(' ')
                headers = {}
                while (h := rf.readline()) not in (b'\r\n', b'\n', b''):   # antetul se termină la linia goală
                    k, _, v = h.decode('latin-1').partition(':')
                    headers[k.strip().lower()] = v.strip()
                body = rf.read(int(headers.get('content-length', 0)))      # corpul delimitat de Content-Length
                self.requests += 1
                if len(parts) != 3 or not parts[2].startswith('HTTP/'):
                    self.reply(c, 400, 'Bad Request', b'bad request line\n', close=True); return
                method, target, version = parts
                close = headers.get('connection', '').lower() == 'close' or version == 'HTTP/1.0'
                self.route(c, method, target, headers, body, close)
                if close:
                    return

    def reply(self, c, code, reason, body=b'', headers=None, close=False, chunks=None):
        h = {'Date': email.utils.formatdate(usegmt=True), 'Server': 'mini-http/0.1'}
        h.update(headers or {})
        if chunks is not None:
            h['Transfer-Encoding'] = 'chunked'
        elif code != 304:
            h['Content-Length'] = str(len(body))
        if close:
            h['Connection'] = 'close'
        head = f'HTTP/1.1 {code} {reason}\r\n' + ''.join(f'{k}: {v}\r\n' for k, v in h.items()) + '\r\n'
        if chunks is not None:
            body = b''.join(f'{len(ch):x}\r\n'.encode() + ch + b'\r\n' for ch in chunks) + b'0\r\n\r\n'
        c.sendall(head.encode('latin-1') + (b'' if code == 304 else body))

    def route(self, c, method, target, headers, body, close):
        KNOWN = {'GET', 'HEAD', 'POST', 'PUT', 'DELETE', 'PATCH', 'OPTIONS'}
        if method not in KNOWN:
            return self.reply(c, 501, 'Not Implemented', b'unknown method\n', close=close)
        if method == 'DELETE':
            return self.reply(c, 405, 'Method Not Allowed', b'pages are never deleted\n', {'Allow': 'GET, HEAD, POST'}, close)
        if target == '/old':
            return self.reply(c, 301, 'Moved Permanently', b'', {'Location': 'https://cn.example.edu/'}, close)
        if target == '/admin':
            who = headers.get('authorization')
            if not who:
                return self.reply(c, 401, 'Unauthorized', b'log in first\n', {'WWW-Authenticate': 'Basic realm="cn"'}, close)
            if who != 'teacher':
                return self.reply(c, 403, 'Forbidden', b'teachers only\n', close=close)
            return self.reply(c, 200, 'OK', b'admin page\n', close=close)
        if target == '/crash':
            return self.reply(c, 500, 'Internal Server Error', b'the application crashed\n', close=close)
        if target == '/stream':                                     # corp de lungime necunoscută → chunked
            return self.reply(c, 200, 'OK', headers={'Content-Type': 'text/plain'}, chunks=[b'Network', b's, week 3'], close=close)
        if target == '/files/notes.pdf':                            # GET condițional
            ims = headers.get('if-modified-since')
            meta = {'Last-Modified': LAST_MOD, 'ETag': '"5f3a-1e2b4"'}
            if ims and email.utils.parsedate_to_datetime(ims) >= LAST_MOD_TS:
                return self.reply(c, 304, 'Not Modified', headers=meta, close=close)
            return self.reply(c, 200, 'OK', b'%PDF-1.7 ... (pretend this is a PDF)\n', {**meta, 'Content-Type': 'application/pdf'}, close)
        if target == '/orders' and method == 'POST':                # POST cu cheie de idempotență
            key = headers.get('idempotency-key')
            if key and key in self.seen_keys:
                return self.reply(c, 200, 'OK', self.seen_keys[key], close=close)
            self.orders.append(body)
            msg = f'order {len(self.orders)} created\n'.encode()
            if key: self.seen_keys[key] = msg
            return self.reply(c, 201, 'Created', msg, {'Location': f'/orders/{len(self.orders)}'}, close)
        if target.startswith('/hello') or target == '/':
            return self.reply(c, 200, 'OK', b'<!doctype html><title>hi</title><p>Hello, week 3!</p>\n',
                              {'Content-Type': 'text/html; charset=utf-8'}, close)
        return self.reply(c, 404, 'Not Found', b'no such object\n', close=close)

    def shutdown(self):
        self.ls.close()

web = MiniHTTPServer()
print('mini-server HTTP pe 127.0.0.1:%d' % web.port)

### Partea clientului: unde se termină un mesaj?
Cititorul de mai jos aplică exact cele trei reguli din curs: **antetul** se termină la prima linie goală; un **corp** are `Content-Length` octeți sau este o succesiune de **bucăți (chunks)** încheiată de o bucată de dimensiune 0 (sau, în lipsa ambelor, se termină când se închide conexiunea).

In [ ]:
def decode_chunked(raw: bytes):
    """Decodează un corp chunked. Întoarce (corpul, numărul de bucăți)."""
    body, pos, n = b'', 0, 0
    while True:
        eol = raw.index(b'\r\n', pos)
        size = int(raw[pos:eol].split(b';')[0], 16)               # dimensiunea în hexazecimal
        pos = eol + 2
        if size == 0:
            return body, n
        body += raw[pos:pos + size]; pos += size + 2; n += 1       # sărim peste CR LF de după date

def read_response(rf, method='GET'):
    """Citește UN răspuns HTTP/1.1 dintr-un fișier socket cu buffer. Întoarce (status, antete, corp, încadrare)."""
    status = rf.readline().decode('latin-1').rstrip('\r\n')
    headers = {}
    while (h := rf.readline()) not in (b'\r\n', b'\n', b''):
        k, _, v = h.decode('latin-1').partition(':'); headers[k.strip().lower()] = v.strip()
    code_ = int(status.split()[1])
    if method == 'HEAD' or code_ in (204, 304) or 100 <= code_ < 200:
        return status, headers, b'', 'fără corp'
    if headers.get('transfer-encoding', '').lower() == 'chunked':
        raw = b''
        while True:
            line = rf.readline(); raw += line
            size = int(line.split(b';')[0], 16)
            if size == 0:
                raw += rf.readline(); break
            raw += rf.read(size + 2)
        return status, headers, decode_chunked(raw)[0], 'chunked'
    if 'content-length' in headers:
        return status, headers, rf.read(int(headers['content-length'])), 'Content-Length'
    return status, headers, rf.read(), 'până la închiderea conexiunii'

def request(method, target, host='localhost', headers=None, body=b'', close=False):
    h = {'Host': host, 'User-Agent': 'course-notebook/1.0', **(headers or {})}
    if body: h['Content-Length'] = str(len(body))
    if close: h['Connection'] = 'close'
    return (f'{method} {target} HTTP/1.1\r\n' + ''.join(f'{k}: {v}\r\n' for k, v in h.items()) + '\r\n').encode() + body

def exchange(sock, rf, method, target, verbose=True, **kw):
    msg = request(method, target, **kw)
    sock.sendall(msg)
    status, headers, body, framing = read_response(rf, method)
    if verbose:
        print('>>> ' + msg.decode('latin-1').replace('\r\n', '\n>>> ').rstrip('> \n'))
        print('<<< ' + status); [print(f'<<< {k}: {v}') for k, v in headers.items()]
        print(f'<<< (corp: {len(body)} octeți, încadrat prin {framing}) {body[:60]!r}\n')
    return status, headers, body

c0, r0 = web.connections, web.requests
s = socket.create_connection(('127.0.0.1', web.port)); rf = s.makefile('rb')
exchange(s, rf, 'GET', '/hello')
exchange(s, rf, 'GET', '/stream')
exchange(s, rf, 'GET', '/files/notes.pdf', close=True)
rf.close(); s.close()      # închidem și obiectul fișier, altfel conexiunea rămâne deschisă
print(f'Statistici server: {web.requests - r0} cereri pe {web.connections - c0} conexiune(i) TCP → conexiune persistentă')

In [ ]:
# Comportamentul HTTP/1.0: o conexiune pentru fiecare obiect (nepersistent)
c0 = web.connections
for target in ('/hello', '/hello?img=1', '/hello?img=2'):
    with socket.create_connection(('127.0.0.1', web.port)) as s:
        s.sendall(f'GET {target} HTTP/1.0\r\nHost: localhost\r\n\r\n'.encode())
        print(f'{target:14s} →', s.makefile('rb').readline().decode().strip())
print(f'3 obiecte au necesitat {web.connections - c0} conexiuni TCP')

In [ ]:
METHODS = [('GET', 'aduce obiectul', True, True), ('HEAD', 'ca GET, doar antetul', True, True),
           ('POST', 'trimite date de procesat', False, False), ('PUT', 'creează sau înlocuiește obiectul', False, True),
           ('DELETE', 'șterge obiectul', False, True), ('PATCH', 'modifică o parte a obiectului', False, False),
           ('OPTIONS', 'ce metode sunt suportate', True, True)]
print(f'{"metoda":8s} {"semnificația":34s} {"sigură":7s} {"idempotentă"}')
for m, what, safe, idem in METHODS:
    print(f'{m:8s} {what:34s} {"da" if safe else "nu":7s} {"da" if idem else "nu"}')

### Un server real
Același cititor funcționează și cu un server web real din Internet (dacă VM-ul Colab îl poate accesa).

In [ ]:
try:
    with socket.create_connection(('example.org', 80), timeout=8) as s:
        rf = s.makefile('rb')
        exchange(s, rf, 'GET', '/', host='example.org')
        exchange(s, rf, 'HEAD', '/', host='example.org', close=True)
except OSError as e:
    print('example.org nu este accesibil de aici:', e)

## A.7 API-ul socket, pas cu pas, și ordinea octeților în rețea
Modulul `socket` din Python este un strat subțire peste aceleași apeluri de sistem ca în C: `socket → bind → listen → accept → recv/send → close` la server, `socket → connect → send/recv → close` la client. Încă un fapt: întregii de pe fir sunt **big-endian**. Dacă uitați `htons` pe o mașină x86 (little-endian), serverul ascultă pe portul greșit:

In [ ]:
print('această mașină este', sys.byteorder, 'endian')
for port in (8080, 5000):
    wrong = struct.unpack('!H', struct.pack('<H', port))[0]    # octeți în ordinea gazdei citiți în ordinea rețelei
    print(f'portul {port} = 0x{port:04X} → fără htons, octeții {struct.pack("<H", port).hex(" ")} sunt citiți ca 0x{wrong:04X} = {wrong}')
print('socket.htons(8080) =', socket.htons(8080))

## A.8 Exercițiile de la sfârșitul cursului, verificate prin cod

In [ ]:
RTT, T = 0.080, 0.010
print('Exercițiul 1 — bază + 4 imagini, RTT 80 ms, 10 ms per obiect')
check('(a) nepersistent (ms)', page_load(4, RTT, T, T) * 1e3, 850)
check('(b) persistent (ms)', page_load(4, RTT, T, T, 'persistent') * 1e3, 530)
check('(c) pipelining (ms)', page_load(4, RTT, T, T, 'pipelined') * 1e3, 290)
print('Exercițiul 2 — 4 conexiuni nepersistente în paralel')
check('timpul de încărcare a paginii (ms)', page_load(4, RTT, T, T, parallel=4) * 1e3, 340)
print('Exercițiul 3 — corp chunked')
body, n = decode_chunked(b'5\r\nHello\r\n8\r\n, world!\r\n0\r\n\r\n')
print(f'   {body!r}, {len(body)} octeți în {n} bucăți'); check('lungimea', len(body), 13)
print('Exercițiul 4 — NU este garantat că un send() ajunge ca un singur recv(), oricare i-ar fi dimensiunea (vezi B.5)')
print('Exercițiul 5 — sin_port = 8080 fără htons')
check('portul real', struct.unpack('!H', struct.pack('<H', 8080))[0], 36895)

**Întrebări de recapitulare**: (1) de ce aplicațiile noi nu necesită modificarea routerelor · (2) clientul și serverul în BitTorrent · (3) 10 000 de conexiuni spre portul 443: cum este livrat un segment? · (4) o aplicație pentru care UDP e mai bun · (5) cum protejează HTTPS o conexiune, la ce strat · (6) partea din URL care nu se trimite niciodată · (7) de ce `Host` este obligatoriu · (8) repetarea unui GET vs. a unui POST · (9) două moduri de a afla sfârșitul unui corp · (10) ce întoarce `accept()`.

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **R1:** 
- **R2:** 
- **R3:** 
- **R4:** 
- **R5:** 
- **R6:** 
- **R7:** 
- **R8:** 
- **R9:** 
- **R10:** 

---
# Partea B · Seminarul 3 — temporizarea HTTP, mesaje și fluxuri de octeți

## B.1 Problema 1 — ce transport?

In [ ]:
P1 = {'a': ('Autentificare la distanță (SSH)', 'TCP'),
      'b': ('Joc de curse, poziții de 30×/s', 'UDP'),
      'c': ('Upload al unui backup de 2 GB', 'TCP'),
      'd': ('Apel video între două telefoane', 'UDP+RTP'),
      'e': ('Interogare DNS', 'UDP'),
      'f': ('Senzor, o citire pe minut, radio cu pierderi', 'UDP')}
MINE = {'a': 'TCP', 'b': '', 'c': '', 'd': '', 'e': '', 'f': ''}   # ← răspunsurile voastre: TCP, UDP sau UDP+RTP
for k, (app, sol) in P1.items():
    print(f"{'✅' if MINE[k].upper() == sol else '❌'} ({k}) {app:42s} → {sol}")

## B.2 Problemele 2 și 3 — cât durează o pagină?
Fișier de bază de 50 KB, 6 imagini de câte 100 KB, legătura de acces de 20 Mbit/s (gâtuirea), RTT 40 ms.

In [ ]:
R, RTT = 20e6, 0.040
tb, ti = 50e3 * 8 / R, 100e3 * 8 / R
check('P2.1 transmisia fișierului de bază (ms)', tb * 1e3, 20); check('P2.1 o imagine (ms)', ti * 1e3, 40)
check('P2.2 nepersistent (ms)', page_load(6, RTT, tb, ti) * 1e3, 820)
check('P2.3 persistent (ms)', page_load(6, RTT, tb, ti, 'persistent') * 1e3, 580)
check('P2.4 pipelining (ms)', page_load(6, RTT, tb, ti, 'pipelined') * 1e3, 380)
t_small = 2e3 * 8 / R
print('P2.5 — 60 de imagini de câte 2 KB (după pagina de bază):')
for mode in ('nonpersistent', 'persistent', 'pipelined'):
    print(f'   {mode:14s} {(page_load(60, RTT, tb, t_small, mode) - (2*RTT + tb)) * 1e3:7.0f} ms')
check('P3.1 șase conexiuni paralele care împart legătura (ms)', page_load(6, RTT, tb, ti, parallel=6, shared_link=True) * 1e3, 420)
print('P3.2 paralelismul ascunde timpii dus-întors, nu transmisia: 6 × 40 ms împart tot o singură legătură de 20 Mbit/s.')

## B.3 Problema 4 — citirea unui schimb HTTP
Schimbul din problemă este analizat prin cod, apoi **reluat** cu mini-serverul din A.6, care răspunde `304 Not Modified` exact ca în fișă.

In [ ]:
REQ = """GET /files/notes.pdf HTTP/1.1
Host: cn.example.edu
User-Agent: curl/8.5.0
If-Modified-Since: Fri, 16 Oct 2026 14:12:30 GMT
Accept: */*
"""
def parse_head(text):
    lines = text.strip().splitlines()
    return lines[0], {k.strip().lower(): v.strip() for k, _, v in (l.partition(':') for l in lines[1:])}

first, hd = parse_head(REQ)
method, target, version = first.split()
print('1. URL        : http://' + hd['host'] + target)
print('2. programul  :', hd['user-agent'], '| persistentă:', version == 'HTTP/1.1' and hd.get('connection') != 'close')
print('3. cere       : trimite obiectul doar dacă a fost modificat după', hd['if-modified-since'])

s = socket.create_connection(('127.0.0.1', web.port)); rf = s.makefile('rb')
status, h, body = exchange(s, rf, 'GET', target, host=hd['host'],
                           headers={'User-Agent': hd['user-agent'], 'If-Modified-Since': hd['if-modified-since'], 'Accept': '*/*'})
print('4. octeți corp:', len(body), '(un 304 nu are niciodată corp; antetul se termină la linia goală)')
print('   și fără If-Modified-Since:'); exchange(s, rf, 'GET', target, close=True)
rf.close(); s.close()      # închidem și obiectul fișier, altfel conexiunea rămâne deschisă

## B.4 Problema 5 — corpuri chunked

In [ ]:
raw = b'1a\r\nabcdefghijklmnopqrstuvwxyz\r\na\r\n0123456789\r\n0\r\n\r\n'
body, n = decode_chunked(raw)
print(body, f'→ {len(body)} octeți în {n} bucăți (0x1a = 26, 0xa = 10)')
check('lungimea', len(body), 36)

## B.5 Problema 6 — ce vede receptorul
Ca să arătăm că **orice împărțire** a fluxului de octeți este posibilă, punem între client și server un mic releu care transmite octeții în bucăți de dimensiuni aleatoare, cu mici pauze. Clientul apelează `send("HELLO")` și `send("WORLD")`; serverul apelează `recv(100)` în mod repetat.

In [ ]:
def resegmenting_relay(target_port, seed):
    """Acceptă o conexiune și îi transmite octeții spre target_port în bucăți de dimensiuni aleatoare."""
    rng = random.Random(seed)
    ls_ = socket.socket(); ls_.bind(('127.0.0.1', 0)); ls_.listen(1)
    def run():
        a, _ = ls_.accept(); data = b''
        a.settimeout(0.3)
        try:
            while chunk := a.recv(100): data += chunk
        except socket.timeout:
            pass
        out = socket.create_connection(('127.0.0.1', target_port))
        out.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
        while data:
            k = rng.randint(1, len(data)); out.sendall(data[:k]); data = data[k:]; time.sleep(0.05)
        out.close(); a.close(); ls_.close()
    threading.Thread(target=run, daemon=True).start()
    return ls_.getsockname()[1]

def receiver():
    ls_ = socket.socket(); ls_.bind(('127.0.0.1', 0)); ls_.listen(1)
    got = []
    def run():
        c, _ = ls_.accept()
        while d := c.recv(100): got.append(d)
        c.close(); ls_.close()
    t = threading.Thread(target=run, daemon=True); t.start()
    return ls_.getsockname()[1], got, t

for seed in (1, 2, 3, 4):
    port, got, t = receiver()
    relay = resegmenting_relay(port, seed)
    c = socket.create_connection(('127.0.0.1', relay)); c.send(b'HELLO'); c.send(b'WORLD'); c.close()
    t.join(5)
    print(f'rularea {seed}: recv a întors {[len(g) for g in got]}  {got}')

In [ ]:
# Două moduri de a face granițele din nou vizibile
def frame_lines(msgs):  return b''.join(m + b'\n' for m in msgs)
def frame_length(msgs): return b''.join(struct.pack('!I', len(m)) + m for m in msgs)

def parse_lines(stream):
    *msgs, rest = stream.split(b'\n'); return msgs, rest

def parse_length(stream):
    msgs, pos = [], 0
    while pos + 4 <= len(stream):
        n = struct.unpack('!I', stream[pos:pos + 4])[0]
        if pos + 4 + n > len(stream): break
        msgs.append(stream[pos + 4:pos + 4 + n]); pos += 4 + n
    return msgs, stream[pos:]

for name, framer, parser in [('separator (linie nouă)', frame_lines, parse_lines), ('prefix de lungime (4 octeți, ordinea rețelei)', frame_length, parse_length)]:
    wire = framer([b'HELLO', b'WORLD'])
    print(f'{name}: {len(wire)} octeți pe fir: {wire.hex(" ")}')
    port, got, t = receiver(); relay = resegmenting_relay(port, 7)
    c = socket.create_connection(('127.0.0.1', relay)); c.sendall(wire); c.close(); t.join(5)
    msgs, rest = parser(b''.join(got))
    print(f'   a sosit în bucățile {[len(g) for g in got]} → mesaje recuperate: {msgs}\n')

## B.6 Problemele 7 și 8 — coduri de stare și cereri repetabile
Fiecare situație din Problema 7 este trimisă mini-serverului; comparați răspunsul lui cu al vostru.

In [ ]:
SITUATIONS = [('situl s-a mutat definitiv pe https', 'GET', '/old', {}),
              ('/admin fără autentificare', 'GET', '/admin', {}),
              ('/admin ca student autentificat', 'GET', '/admin', {'Authorization': 'student'}),
              ('DELETE /index.html', 'DELETE', '/index.html', {}),
              ('cont nou creat prin POST', 'POST', '/orders', {}),
              ('aplicația s-a prăbușit', 'GET', '/crash', {}),
              ('linia de cerere „GTE / HTTP/1.1”', 'GTE', '/', {})]
EXPECTED = ['301', '401', '403', '405', '201', '500', '501']
for (what, m, tgt, hdrs), exp in zip(SITUATIONS, EXPECTED):
    with socket.create_connection(('127.0.0.1', web.port)) as s:
        s.sendall(request(m, tgt, headers=hdrs, body=b'x' if m == 'POST' else b'', close=True))
        st = s.makefile('rb').readline().decode().strip()
    print(f"{'✅' if st.split()[1] == exp else '❌'} {what:34s} → {st}")
print('(Pentru GTE este acceptabil și 400 Bad Request.)')

In [ ]:
P8 = [('GET /timetable?group=CS2', True, 'sigură și idempotentă'),
      ('PUT /profile/photo', True, 'idempotentă: aceeași fotografie de două ori, același rezultat'),
      ('DELETE /cart/items/17', True, 'idempotentă: a doua nu mai găsește nimic (404 e în regulă)'),
      ('POST /orders', False, 'NU este idempotentă: comanda poate exista deja')]
for req_, ok, why in P8:
    print(f'{req_:28s} se repetă automat? {"da" if ok else "NU"}  — {why}')

print('\nSoluția pentru POST: o cheie de idempotență. Răspunsul la primul POST se „pierde”; aplicația reîncearcă:')
n0 = len(web.orders)
for attempt in (1, 2):
    with socket.create_connection(('127.0.0.1', web.port)) as s:
        s.sendall(request('POST', '/orders', headers={'Idempotency-Key': 'cart-7f3a'}, body=b'2 x notebook', close=True))
        st, _, bd, _ = read_response(s.makefile('rb'), 'POST')
    print(f'   încercarea {attempt}: {st} {bd.decode().strip()}{"   (răspuns pierdut)" if attempt == 1 else ""}')
print(f'   comenzi create efectiv: {len(web.orders) - n0}')

## B.7 Problema 9 — socket-uri și 4-tupluri pe un server aglomerat
Două conexiuni de la un „laptop” (127.0.0.1) și una de la un „telefon” (îi legăm socket-ul de 127.0.0.2, o altă adresă de loopback).

In [ ]:
srv = socket.socket(); srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1); srv.bind(('127.0.0.1', 0)); srv.listen(8)
port = srv.getsockname()[1]
laptop = [socket.create_connection(('127.0.0.1', port)) for _ in range(2)]
phone = socket.socket(); phone.bind(('127.0.0.2', 0)); phone.connect(('127.0.0.1', port))
accepted = [srv.accept()[0] for _ in range(3)]
print(f'Socket-urile serverului: 1 de ascultare + {len(accepted)} conectate = {1 + len(accepted)}')
for c in accepted:
    print('   4-tuplu', c.getpeername() + c.getsockname())
sh(f'ss -tn state established "( sport = :{port} )"')
for x in laptop + [phone] + accepted + [srv]: x.close()

## B.8 Problema 10 — proiectarea și implementarea unui protocol de upload de fișiere
Răspunsul din seminar, implementat: **cererea** `PUT <nume> <dimensiune>\r\n` urmată de exact `<dimensiune>` octeți bruți; **răspunsul** `OK\r\n` sau `ERR <cod> <text>\r\n`. Antetul este delimitat printr-un separator, corpul prin lungimea sa — cele două reguli ale HTTP/1.1. Fișierul de test conține intenționat caractere de linie nouă și octeți zero.

In [ ]:
class UploadServer(threading.Thread):
    MAX = 10_000_000
    def __init__(self):
        super().__init__(daemon=True)
        self.ls = socket.socket(); self.ls.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        self.ls.bind(('127.0.0.1', 0)); self.ls.listen(8); self.port = self.ls.getsockname()[1]
        self.files, self.log = {}, []
        self.start()
    def run(self):
        while True:
            try: c, _ = self.ls.accept()
            except OSError: return
            with c:
                rf = c.makefile('rb')
                while (line := rf.readline()):
                    if not line.endswith(b'\r\n'):
                        self.log.append('conexiunea s-a închis în mijlocul unei linii de antet'); break
                    parts = line[:-2].decode('ascii', 'replace').split(' ')
                    if len(parts) != 3 or parts[0] != 'PUT' or not parts[2].isdigit():
                        c.sendall(b'ERR 400 bad request\r\n'); continue
                    name, size = parts[1], int(parts[2])
                    if size > self.MAX:
                        c.sendall(b'ERR 413 too large\r\n'); break
                    data = rf.read(size)
                    if len(data) < size:            # temă: conexiunea s-a închis înainte de <dimensiune> octeți
                        self.log.append(f'{name}: au sosit doar {len(data)} din {size} octeți → fișierul parțial este aruncat, niciun răspuns posibil')
                        break
                    self.files[name] = data
                    c.sendall(b'OK\r\n')

def upload(sock, rf, name, data):
    sock.sendall(f'PUT {name} {len(data)}\r\n'.encode() + data)
    return rf.readline().decode().strip()

up = UploadServer()
payload = b'line 1\nline 2\r\n\x00\x00binary\x00\xff' * 1000
s = socket.create_connection(('127.0.0.1', up.port)); rf = s.makefile('rb')
print('upload 1  :', upload(s, rf, 'data.bin', payload))
print('upload 2  :', upload(s, rf, 'small.txt', b'hi\n'), ' (aceeași conexiune)')
print('cerere rea:', (s.sendall(b'GET data.bin\r\n'), rf.readline().decode().strip())[1])
print('repetă 1  :', upload(s, rf, 'data.bin', payload), ' (PUT este idempotent: același rezultat)')
rf.close(); s.close(); time.sleep(0.2)
print('stocat intact:', up.files['data.bin'] == payload, f'({len(payload)} octeți)')

# Temă: clientul dispare în mijlocul corpului
s = socket.create_connection(('127.0.0.1', up.port))
s.sendall(b'PUT broken.bin 1000\r\n' + b'x' * 300); s.close(); time.sleep(0.3)
print('jurnalul serverului:', up.log[-1] if up.log else '(nimic înregistrat încă)')

**Temă**: exercițiile 1–5 de la curs (vezi A.8); comparați `curl -v` și `curl -v --http1.0` (mai jos); ce ar trebui să facă serverul din Problema 10 dacă conexiunea se închide prea devreme (vezi ultima linie de mai sus).

In [ ]:
for opt in ('', '--http1.0'):
    print(f'### curl -v {opt} http://example.org/')
    out = sh(f'curl -sv {opt} http://example.org/ -o /dev/null 2>&1 | grep -E "^[<>] |^\\* (Connection|Closing|shutting)" | head -20', show=False)
    print(out or '(inaccesibil)')

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Tema 2 — diferențele dintre cererile HTTP/1.1 și HTTP/1.0 și comportamentul conexiunii:** 

---
# Partea C · Laboratorul 3 (Linux) — Socket-uri I: un client și un server TCP în C
Celulele `%%writefile` de mai jos creează fișierele din arhiva laboratorului în dosarul `lab3/`. Citiți fiecare program înainte de a-l rula — comentariile urmează pașii numerotați din Cursul 3, Figura 2. Numere de port: 5000–5004, ca în fișă.

### Fișierele

In [ ]:
%%writefile lab3/netio.h
/* netio.h - helpers for TCP byte streams (Computer Networks, Lab 3) */
#ifndef NETIO_H
#define NETIO_H
#include <stdint.h>
#include <errno.h>
#include <unistd.h>
#include <arpa/inet.h>
#include <sys/socket.h>

/* Write exactly len bytes. Returns 0 on success, -1 on error. */
static inline int send_all(int fd, const void *buf, size_t len) {
    const char *p = buf;
    while (len > 0) {
        ssize_t n = send(fd, p, len, 0);
        if (n < 0) { if (errno == EINTR) continue; return -1; }
        p += n; len -= (size_t)n;
    }
    return 0;
}

/* Read exactly len bytes. Returns 1 on success, 0 if the peer closed first, -1 on error. */
static inline int recv_all(int fd, void *buf, size_t len) {
    char *p = buf;
    while (len > 0) {
        ssize_t n = recv(fd, p, len, 0);
        if (n < 0) { if (errno == EINTR) continue; return -1; }
        if (n == 0) return 0;
        p += n; len -= (size_t)n;
    }
    return 1;
}

/* Length-prefixed messages: 4-byte length in network byte order, then the payload. */
static inline int send_msg(int fd, const void *buf, uint32_t len) {
    uint32_t netlen = htonl(len);
    if (send_all(fd, &netlen, sizeof netlen) < 0) return -1;
    return send_all(fd, buf, len);
}

/* Returns the payload length, 0 if the peer closed, -1 on error or if the message exceeds cap. */
static inline long recv_msg(int fd, void *buf, uint32_t cap) {
    uint32_t netlen;
    int r = recv_all(fd, &netlen, sizeof netlen);
    if (r <= 0) return r;
    uint32_t len = ntohl(netlen);
    if (len > cap) return -1;
    r = recv_all(fd, buf, len);
    return r <= 0 ? -1 : (long)len;
}
#endif

In [ ]:
%%writefile lab3/echo_client.c
/* echo_client.c - send each line of standard input to a TCP server, print what comes back.
   Usage: ./echo_client HOST PORT */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netdb.h>
#include <sys/socket.h>
#include "netio.h"

int main(int argc, char *argv[]) {
    if (argc != 3) { fprintf(stderr, "usage: %s HOST PORT\n", argv[0]); return 1; }
    setvbuf(stdout, NULL, _IOLBF, 0);
    struct addrinfo hints = {0}, *res, *ai;
    hints.ai_family = AF_UNSPEC;          /* IPv4 or IPv6 */
    hints.ai_socktype = SOCK_STREAM;      /* TCP */
    int err = getaddrinfo(argv[1], argv[2], &hints, &res);
    if (err) { fprintf(stderr, "getaddrinfo: %s\n", gai_strerror(err)); return 1; }

    int fd = -1;
    for (ai = res; ai != NULL; ai = ai->ai_next) {          /* try each address in turn */
        fd = socket(ai->ai_family, ai->ai_socktype, ai->ai_protocol);
        if (fd < 0) continue;
        if (connect(fd, ai->ai_addr, ai->ai_addrlen) == 0) break;
        close(fd); fd = -1;
    }
    freeaddrinfo(res);
    if (fd < 0) { perror("connect"); return 1; }

    char line[1024], reply[1024];
    while (fgets(line, sizeof line, stdin) != NULL) {
        size_t len = strlen(line);
        if (send_all(fd, line, len) < 0) { perror("send"); break; }
        /* The echo has the same length as what we sent, but may arrive in pieces. */
        int r = recv_all(fd, reply, len);
        if (r <= 0) { fprintf(stderr, r == 0 ? "server closed the connection\n" : "recv failed\n"); break; }
        fwrite(reply, 1, len, stdout);
    }
    close(fd);
    return 0;
}

In [ ]:
%%writefile lab3/echo_server.c
/* echo_server.c - iterative TCP echo server: one client at a time.
   Usage: ./echo_server PORT */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <arpa/inet.h>
#include <netinet/in.h>
#include <sys/socket.h>
#include "netio.h"

int main(int argc, char *argv[]) {
    if (argc != 2) { fprintf(stderr, "usage: %s PORT\n", argv[0]); return 1; }
    setvbuf(stdout, NULL, _IOLBF, 0);                         /* print each line at once */
    int lfd = socket(AF_INET, SOCK_STREAM, 0);               /* 1. create */
    if (lfd < 0) { perror("socket"); return 1; }
    int on = 1;
    setsockopt(lfd, SOL_SOCKET, SO_REUSEADDR, &on, sizeof on); /* allow quick restarts */
    struct sockaddr_in addr = {0};
    addr.sin_family = AF_INET;
    addr.sin_addr.s_addr = htonl(INADDR_ANY);                 /* all local addresses */
    addr.sin_port = htons((uint16_t)atoi(argv[1]));            /* host -> network byte order */
    if (bind(lfd, (struct sockaddr *)&addr, sizeof addr) < 0) { perror("bind"); return 1; } /* 2. bind */
    if (listen(lfd, 8) < 0) { perror("listen"); return 1; }  /* 3. listen, backlog 8 */
    printf("listening on port %s\n", argv[1]);

    for (;;) {
        struct sockaddr_in peer; socklen_t plen = sizeof peer;
        int cfd = accept(lfd, (struct sockaddr *)&peer, &plen); /* 4. accept: a NEW socket */
        if (cfd < 0) { perror("accept"); continue; }
        char ip[INET_ADDRSTRLEN];
        inet_ntop(AF_INET, &peer.sin_addr, ip, sizeof ip);
        printf("connection from %s:%u\n", ip, ntohs(peer.sin_port));
        char buf[65536]; ssize_t n; long total = 0;
        while ((n = recv(cfd, buf, sizeof buf, 0)) > 0) {      /* 5. read what has arrived */
            printf("  recv returned %zd bytes\n", n);
            total += n;
            if (send_all(cfd, buf, (size_t)n) < 0) { perror("send"); break; }
        }
        printf("client closed after %ld bytes\n", total);
        close(cfd);                                            /* 6. close this connection */
    }
}

In [ ]:
%%writefile lab3/upper_server.c
/* upper_server.c - line-based protocol: each line ending in \n comes back in upper case.
   Usage: ./upper_server PORT */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>
#include <unistd.h>
#include <netinet/in.h>
#include <sys/socket.h>
#include "netio.h"

int main(int argc, char *argv[]) {
    if (argc != 2) { fprintf(stderr, "usage: %s PORT\n", argv[0]); return 1; }
    setvbuf(stdout, NULL, _IOLBF, 0);
    int lfd = socket(AF_INET, SOCK_STREAM, 0);
    if (lfd < 0) { perror("socket"); return 1; }
    int on = 1;
    setsockopt(lfd, SOL_SOCKET, SO_REUSEADDR, &on, sizeof on);
    struct sockaddr_in addr = {0};
    addr.sin_family = AF_INET;
    addr.sin_addr.s_addr = htonl(INADDR_ANY);
    addr.sin_port = htons((uint16_t)atoi(argv[1]));
    if (bind(lfd, (struct sockaddr *)&addr, sizeof addr) < 0) { perror("bind"); return 1; }
    if (listen(lfd, 8) < 0) { perror("listen"); return 1; }
    printf("listening on port %s\n", argv[1]);

    for (;;) {
        int cfd = accept(lfd, NULL, NULL);
        if (cfd < 0) continue;
        char buf[4096]; size_t used = 0; ssize_t n;
        while ((n = recv(cfd, buf + used, sizeof buf - used, 0)) > 0) {
            used += (size_t)n;
            char *nl;
            /* A single recv may hold part of a line, one line, or several lines. */
            while ((nl = memchr(buf, '\n', used)) != NULL) {
                size_t linelen = (size_t)(nl - buf) + 1;
                for (size_t i = 0; i < linelen; i++) buf[i] = (char)toupper((unsigned char)buf[i]);
                if (send_all(cfd, buf, linelen) < 0) break;
                memmove(buf, buf + linelen, used - linelen);   /* keep the rest for later */
                used -= linelen;
            }
            if (used == sizeof buf) { const char *e = "ERR line too long\n"; send_all(cfd, e, strlen(e)); break; }
        }
        if (used > 0) printf("connection closed with %zu bytes of an incomplete line discarded\n", used);
        close(cfd);
    }
}

In [ ]:
%%writefile lab3/blast.c
/* blast.c - send N bytes with ONE call to send_all, close the sending direction, then read the echo.
   Usage: ./blast HOST PORT N */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netdb.h>
#include <sys/socket.h>
#include "netio.h"

int main(int argc, char *argv[]) {
    if (argc != 4) { fprintf(stderr, "usage: %s HOST PORT N\n", argv[0]); return 1; }
    size_t n = (size_t)strtoul(argv[3], NULL, 10);
    struct addrinfo hints = {0}, *res;
    hints.ai_family = AF_UNSPEC; hints.ai_socktype = SOCK_STREAM;
    int err = getaddrinfo(argv[1], argv[2], &hints, &res);
    if (err) { fprintf(stderr, "getaddrinfo: %s\n", gai_strerror(err)); return 1; }
    int fd = socket(res->ai_family, res->ai_socktype, res->ai_protocol);
    if (fd < 0 || connect(fd, res->ai_addr, res->ai_addrlen) < 0) { perror("connect"); return 1; }
    freeaddrinfo(res);

    char *data = malloc(n ? n : 1);
    if (!data) { perror("malloc"); return 1; }
    memset(data, 'x', n);
    if (send_all(fd, data, n) < 0) { perror("send"); return 1; }   /* ONE call for all N bytes */
    shutdown(fd, SHUT_WR);                                          /* "I have finished sending" */

    char buf[65536]; ssize_t r; size_t back = 0;                    /* drain the echo */
    while ((r = recv(fd, buf, sizeof buf, 0)) > 0) back += (size_t)r;
    printf("sent %zu bytes in one send_all, received %zu bytes back\n", n, back);
    free(data); close(fd);
    return 0;
}

In [ ]:
%%writefile lab3/msg_server.c
/* msg_server.c - length-prefixed messages: replies "OK <n>" (as a message) to every message of n bytes.
   Usage: ./msg_server PORT */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netinet/in.h>
#include <sys/socket.h>
#include "netio.h"

#define CAP (1u << 20)                                   /* largest message accepted: 1 MiB */

int main(int argc, char *argv[]) {
    if (argc != 2) { fprintf(stderr, "usage: %s PORT\n", argv[0]); return 1; }
    setvbuf(stdout, NULL, _IOLBF, 0);
    int lfd = socket(AF_INET, SOCK_STREAM, 0);
    if (lfd < 0) { perror("socket"); return 1; }
    int on = 1;
    setsockopt(lfd, SOL_SOCKET, SO_REUSEADDR, &on, sizeof on);
    struct sockaddr_in addr = {0};
    addr.sin_family = AF_INET;
    addr.sin_addr.s_addr = htonl(INADDR_ANY);
    addr.sin_port = htons((uint16_t)atoi(argv[1]));
    if (bind(lfd, (struct sockaddr *)&addr, sizeof addr) < 0) { perror("bind"); return 1; }
    if (listen(lfd, 8) < 0) { perror("listen"); return 1; }
    printf("listening on port %s\n", argv[1]);

    char *buf = malloc(CAP);
    if (!buf) { perror("malloc"); return 1; }
    for (;;) {
        int cfd = accept(lfd, NULL, NULL);
        if (cfd < 0) continue;
        long len;
        while ((len = recv_msg(cfd, buf, CAP)) > 0) {
            printf("message of %ld bytes\n", len);
            char reply[32];
            int rl = snprintf(reply, sizeof reply, "OK %ld", len);
            if (send_msg(cfd, reply, (uint32_t)rl) < 0) break;
        }
        if (len < 0) printf("bad or too long message (limit %u bytes): closing\n", CAP);
        else printf("client closed\n");
        close(cfd);
    }
}

In [ ]:
%%writefile lab3/msg_client.c
/* msg_client.c - send each argument as a length-prefixed message, ALL before reading any reply.
   Usage: ./msg_client HOST PORT MSG... */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netdb.h>
#include <sys/socket.h>
#include "netio.h"

int main(int argc, char *argv[]) {
    if (argc < 4) { fprintf(stderr, "usage: %s HOST PORT MSG...\n", argv[0]); return 1; }
    struct addrinfo hints = {0}, *res;
    hints.ai_family = AF_UNSPEC; hints.ai_socktype = SOCK_STREAM;
    int err = getaddrinfo(argv[1], argv[2], &hints, &res);
    if (err) { fprintf(stderr, "getaddrinfo: %s\n", gai_strerror(err)); return 1; }
    int fd = socket(res->ai_family, res->ai_socktype, res->ai_protocol);
    if (fd < 0 || connect(fd, res->ai_addr, res->ai_addrlen) < 0) { perror("connect"); return 1; }
    freeaddrinfo(res);

    for (int i = 3; i < argc; i++)                                   /* send them all first */
        if (send_msg(fd, argv[i], (uint32_t)strlen(argv[i])) < 0) { perror("send"); return 1; }
    for (int i = 3; i < argc; i++) {                                 /* then read the replies */
        char reply[64];
        long n = recv_msg(fd, reply, sizeof reply - 1);
        if (n <= 0) { fprintf(stderr, "no reply\n"); break; }
        reply[n] = '\0';
        printf("%s -> %s\n", argv[i], reply);
    }
    close(fd);
    return 0;
}

In [ ]:
%%writefile lab3/file_client.c
/* file_client.c - send a whole file as ONE length-prefixed message and print the server's reply.
   Usage: ./file_client HOST PORT FILE */
#include <stdio.h>
#include <stdlib.h>
#include <unistd.h>
#include <netdb.h>
#include <sys/socket.h>
#include <sys/stat.h>
#include "netio.h"

int main(int argc, char *argv[]) {
    if (argc != 4) { fprintf(stderr, "usage: %s HOST PORT FILE\n", argv[0]); return 1; }
    FILE *f = fopen(argv[3], "rb");
    if (!f) { perror("fopen"); return 1; }
    struct stat st;
    if (fstat(fileno(f), &st) < 0) { perror("stat"); return 1; }
    size_t size = (size_t)st.st_size;
    char *data = malloc(size ? size : 1);
    if (!data || fread(data, 1, size, f) != size) { fprintf(stderr, "cannot read file\n"); return 1; }
    fclose(f);

    struct addrinfo hints = {0}, *res;
    hints.ai_family = AF_UNSPEC; hints.ai_socktype = SOCK_STREAM;
    int err = getaddrinfo(argv[1], argv[2], &hints, &res);
    if (err) { fprintf(stderr, "getaddrinfo: %s\n", gai_strerror(err)); return 1; }
    int fd = socket(res->ai_family, res->ai_socktype, res->ai_protocol);
    if (fd < 0 || connect(fd, res->ai_addr, res->ai_addrlen) < 0) { perror("connect"); return 1; }
    freeaddrinfo(res);

    if (send_msg(fd, data, (uint32_t)size) < 0) perror("send");
    char reply[64];
    long n = recv_msg(fd, reply, sizeof reply - 1);
    if (n > 0) { reply[n] = '\0'; printf("file of %zu bytes -> server says: %s\n", size, reply); }
    else printf("file of %zu bytes -> no reply (the server closed the connection)\n", size);
    free(data); close(fd);
    return 0;
}

In [ ]:
%%writefile lab3/broken_pipe.c
/* broken_pipe.c - keep writing to a connection after the server has closed it.
   Usage: ./broken_pipe HOST PORT [ignore]   ("ignore" sets SIGPIPE to SIG_IGN) */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <signal.h>
#include <unistd.h>
#include <netdb.h>
#include <sys/socket.h>

int main(int argc, char *argv[]) {
    if (argc < 3) { fprintf(stderr, "usage: %s HOST PORT [ignore]\n", argv[0]); return 1; }
    setvbuf(stdout, NULL, _IONBF, 0);
    if (argc == 4 && strcmp(argv[3], "ignore") == 0) signal(SIGPIPE, SIG_IGN);
    struct addrinfo hints = {0}, *res;
    hints.ai_family = AF_UNSPEC; hints.ai_socktype = SOCK_STREAM;
    if (getaddrinfo(argv[1], argv[2], &hints, &res)) { fprintf(stderr, "getaddrinfo failed\n"); return 1; }
    int fd = socket(res->ai_family, res->ai_socktype, res->ai_protocol);
    if (fd < 0 || connect(fd, res->ai_addr, res->ai_addrlen) < 0) { perror("connect"); return 1; }
    freeaddrinfo(res);
    for (int i = 1; i <= 5; i++) {
        sleep(1);                                    /* meanwhile the server closes the connection */
        ssize_t n = send(fd, "data\n", 5, 0);
        if (n < 0) { perror("send"); return 1; }
        printf("send %d returned %zd\n", i, n);
    }
    close(fd);
    return 0;
}

In [ ]:
%%writefile lab3/Makefile
CC = gcc
CFLAGS = -Wall -Wextra -O2
PROGS = echo_client echo_server upper_server blast msg_server msg_client file_client broken_pipe

all: $(PROGS)

%: %.c netio.h
	$(CC) $(CFLAGS) -o $@ $<

clean:
	rm -f $(PROGS)

In [ ]:
sh(f'cd {LAB} && make clean > /dev/null; make')
sh(f'ls -l {LAB}')

## Sarcina 1 — Un server fără să scrieți cod
`nc -l 5000` ascultă; `nc localhost 5000` se conectează. Notebook-ul joacă rolul ambelor terminale.

In [ ]:
Bg.stop_all(); sh('pkill -f "nc -l 5000"', show=False)
A = Bg('nc -l 5000')                              # terminalul A
sh('ss -tln | grep -E "State|:5000"')             # terminalul C, înainte de client
B = Bg('nc -N localhost 5000')                    # terminalul B   (-N: trimite FIN când se termină stdin)
sh('ss -tn | grep -E "State|:5000"')              # terminalul C, după
B.write('hello from B\n'); A.write('hello from A\n'); time.sleep(0.5)
show('terminalul A (server) afișează', A.output()); show('terminalul B (client) afișează', B.output())

In [ ]:
B.eof(); time.sleep(1)                            # Ctrl+D în terminalul B
print('clientul mai rulează:', B.running(), '| serverul mai rulează:', A.running())
sh('ss -tan | grep -E "State|:5000"')
A.stop(); B.stop()

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q1 — ss -tln înainte, ss -tn după; 4-tuplul; cine a ales portul clientului?:** 
- **Q2 — ce se întâmplă cu serverul după Ctrl+D; ce mesaj TCP l-a anunțat?:** 

## Sarcina 2 — Clientul echo (întâi cu `nc`, apoi fără server)

In [ ]:
A = Bg('nc -l 5000')
C = Bg('./echo_client localhost 5000')
C.write('ping\n'); time.sleep(0.4)
show('nc (server) a primit', A.output())
A.write('ping\n'); time.sleep(0.4)                # facem ecoul de mână
show('echo_client a afișat', C.output())
C.eof(); A.stop(); C.stop()

In [ ]:
# Q4 — nimic nu ascultă pe portul 5000
sh(f'cd {LAB} && echo test | ./echo_client localhost 5000; echo "cod de ieșire $?"')

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q3 — de ce recv_all(fd, reply, len) și nu un singur recv?:** 
- **Q4 — eroarea și mesajul TCP care a provocat-o:** 

## Sarcina 3 — Serverul echo (iterativ: câte un client pe rând)

In [ ]:
S = Bg('./echo_server 5000')
B = Bg('./echo_client localhost 5000'); B.write('first client, line 1\n')
C = Bg('./echo_client localhost 5000'); C.write('second client waits\n')
time.sleep(1)
show('clientul B', B.output()); show('clientul C (încă fără ecou!)', C.output())
sh('ss -tn | grep -E "State|:5000"')              # este conexiunea lui C stabilită?
B.eof(); time.sleep(1)                            # B iese
show('clientul C după ce iese B', C.output())
C.eof(); time.sleep(0.5)
show('jurnalul serverului', S.output())

In [ ]:
# Q7 — eliminăm htons din linia care setează sin_port
sh(f"cd {LAB} && sed 's/htons((uint16_t)atoi(argv\\[1\\]))/(uint16_t)atoi(argv[1])/' echo_server.c > echo_server_nohtons.c "
   f"&& grep -n 'sin_port' echo_server_nohtons.c && gcc -Wall -O2 -o echo_server_nohtons echo_server_nohtons.c")
S.stop()
N = Bg('./echo_server_nohtons 5000')
show('afișează', N.output())
sh('ss -tlnp | grep -E "State|echo_server"')          # priviți coloana Local Address:Port
print('Explicație: 5000 =', f'0x{5000:04X}', '→ stocat ca octeții', struct.pack('<H', 5000).hex(' '),
      '→ citit în ordinea rețelei ca', struct.unpack('!H', struct.pack('<H', 5000))[0])
N.stop()

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q5 — pașii serverului vs. Figura 2; ce afișează; de unde vine portul clientului:** 
- **Q6 — de ce așteaptă al doilea client; este conexiunea lui stabilită?:** 
- **Q7 — portul real fără htons, explicat:** 

## Sarcina 4 — Socket-urile așa cum le vede sistemul de operare

In [ ]:
S = Bg('./echo_server 5000'); B = Bg('./echo_client localhost 5000'); B.write('x\n'); time.sleep(0.4)
sh('ss -tnp | grep -E "5000|echo"')
sh(f'ls -l /proc/{S.pid}/fd')
B.eof(); B.stop()

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q8 — câte socket-uri are echo_server cu un client, în ce stări, ce descriptori:** 

## Sarcina 5 — Un flux nu este o succesiune de mesaje
`blast` trimite N octeți cu **un singur** `send_all` și își închide direcția de trimitere. De câte apeluri `recv` are nevoie serverul?

In [ ]:
S.new_output()
def recv_sizes(log):
    return [int(x) for x in re.findall(r'recv returned (\d+)', log)]

results = []
for n in (10, 5000, 300000, 300000, 300000, 3000000):
    out = sh(f'cd {LAB} && ./blast localhost 5000 {n}', show=False).strip()
    time.sleep(0.3)
    sizes = recv_sizes(S.new_output())
    results.append((n, sizes))
    print(f'N = {n:8d}: {len(sizes):3d} apeluri recv  dimensiuni {sizes[:8]}{" ..." if len(sizes) > 8 else ""}')
print('\n', out)

In [ ]:
big = [s for n, sz in results if n == 3000000 for s in sz]
plt.hist(big, bins=30); plt.xlabel('octeți întorși de un recv()'); plt.ylabel('număr')
plt.title('Un send_all de 3 000 000 de octeți, așa cum l-a primit serverul'); plt.show()
print('Cel mai mare recv posibil aici:', 'sizeof buf = 65536 de octeți; limitat și de cât a sosit în bufferul socket-ului')
sh('sysctl net.ipv4.tcp_rmem net.ipv4.tcp_wmem')

In [ ]:
# Q10 — bufferul serverului redus la 7 octeți
S.stop()
sh(f"cd {LAB} && sed 's/char buf\\[65536\\]/char buf[7]/' echo_server.c > echo_server7.c && gcc -Wall -O2 -o echo_server7 echo_server7.c")
S7 = Bg('./echo_server7 5000')
B = Bg('./echo_client localhost 5000'); B.write('hello world\n'); time.sleep(0.5)
show('serverul cu un buffer de 7 octeți', S7.output()); show('clientul', B.output())
B.eof(); S7.stop(); B.stop()

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q9 — un send, câte recv-uri? Aceleași dimensiuni de fiecare dată? Ce limitează un recv?:** 
- **Q10 — ieșirea serverului cu un buffer de 7 octeți; de ce clientul primește totuși ecoul complet:** 

## Sarcina 6 — Încadrarea cu linii (`upper_server`)

In [ ]:
U = Bg('./upper_server 5001')
sh(f"printf 'abc\\ndef\\nghi' | nc -N localhost 5001")
time.sleep(0.3); show('jurnalul serverului', U.output())

In [ ]:
# Q12 — o linie mai lungă decât bufferul de 4096 de octeți al serverului
reply = b''
s = socket.create_connection(('127.0.0.1', 5001))
try:
    s.sendall(b'a' * 5000 + b'\n')
    time.sleep(0.3)
    while d := s.recv(65536): reply += d
except OSError as e:
    print(f'{type(e).__name__}: serverul a închis când o parte din octeții noștri erau încă necitiți, așa că TCP-ul lui a răspuns cu RST')
finally:
    s.close()
print('primit:', reply if reply else '(nimic — RST-ul poate distruge și linia ERR)')
U.stop()

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q11 — de ce „ghi” nu primește niciodată răspuns; ce ar trebui să spună un protocol despre o ultimă linie incompletă:** 
- **Q12 — comportamentul cu o linie mai lungă de 4096 de octeți; este bun; un design mai bun:** 

## Sarcina 7 — Încadrarea cu prefix de lungime

In [ ]:
MS = Bg('./msg_server 5002')
sh(f'cd {LAB} && ./msg_client localhost 5002 short "a longer message" x')
wire = b''.join(struct.pack('!I', len(m)) + m for m in (b'short', b'a longer message', b'x'))
print(f'Clientul a trimis {len(wire)} octeți:'); print(' ', wire.hex(' '))

In [ ]:
# Q14 — un fișier întreg ca un singur mesaj (file_client.c este o soluție model)
sh(f'cd {LAB} && head -c 200000 /dev/urandom > small.bin && head -c 2000000 /dev/urandom > big.bin && stat -c "%n %s bytes" small.bin big.bin')
sh(f'cd {LAB} && ./file_client localhost 5002 small.bin')
sh(f'cd {LAB} && ./file_client localhost 5002 big.bin')
time.sleep(0.3); show('jurnalul msg_server', MS.output()); MS.stop()

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q13 — cum găsește serverul sfârșitul fiecărui mesaj; de ce htonl:** 
- **Q14 — dimensiunea fișierului vs. dimensiunea raportată; ce se întâmplă peste 1 MB și de ce:** 

## Sarcina 8 — Stricăm lucrurile intenționat

In [ ]:
print('1) Conexiune refuzată (Connection refused)')
sh(f'cd {LAB} && echo x | ./echo_client localhost 5999')

print('2) Adresă deja folosită (Address already in use: server fără SO_REUSEADDR, repornit imediat)')
sh(f"cd {LAB} && grep -v -E 'SO_REUSEADDR|int on = 1' echo_server.c > echo_server_noreuse.c && gcc -Wall -O2 -o echo_server_noreuse echo_server_noreuse.c")
S = Bg('./echo_server_noreuse 5003'); B = Bg('./echo_client localhost 5003'); B.write('x\n'); time.sleep(0.3)
S.stop(signal.SIGINT); B.eof(); time.sleep(0.5)          # Ctrl+C pe server cât timp un client este conectat
sh('ss -tan | grep -E "State|:5003"')
S2 = Bg('./echo_server_noreuse 5003'); show('repornire', S2.output()); S2.stop(); B.stop()

In [ ]:
print('3) Peer închis: oprim clientul cât timp serverul așteaptă în recv')
S = Bg('./echo_server 5000'); B = Bg('./echo_client localhost 5000'); B.write('abc\n'); time.sleep(0.3)
B.stop(signal.SIGINT); time.sleep(0.4); show('serverul', S.output())

print('\n4) Broken pipe: scriere după ce serverul a închis conexiunea')
closer = socket.socket(); closer.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1); closer.bind(('127.0.0.1', 5004)); closer.listen(4)
def accept_and_close():
    for _ in range(2):
        c, _ = closer.accept(); c.close()                  # serverul închide imediat
threading.Thread(target=accept_and_close, daemon=True).start()
r = subprocess.run(f'cd {LAB} && ./broken_pipe localhost 5004', shell=True, capture_output=True, text=True)
print(r.stdout + r.stderr, f'→ cod de retur {r.returncode}' + ('  (oprit de semnalul 13 = SIGPIPE)' if r.returncode in (-13, 141) else ''))
r = subprocess.run(f'cd {LAB} && ./broken_pipe localhost 5004 ignore', shell=True, capture_output=True, text=True)
print(r.stdout + r.stderr, f'→ cu SIGPIPE ignorat: cod de retur {r.returncode}')
closer.close(); Bg.stop_all()

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q15 — cele patru texte de eroare și mecanismul TCP din spatele fiecăruia:** 
- **Q16 — de ce SO_REUSEADDR în dezvoltare; este sigur în producție?:** 

## Aceleași programe în Python
`sendall` este `send_all`; comportamentul fluxului — și nevoia de `recv_all` — sunt identice.

In [ ]:
%%writefile lab3/python/echo_server.py
"""echo_server.py - iterative TCP echo server (Linux and Windows). Usage: python echo_server.py PORT"""
import socket, sys
port = int(sys.argv[1])
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as ls:
    ls.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    ls.bind(("127.0.0.1", port))
    ls.listen(8)
    print(f"listening on 127.0.0.1:{port}", flush=True)
    while True:
        conn, (ip, cport) = ls.accept()          # a NEW socket for this client
        print(f"connection from {ip}:{cport}", flush=True)
        with conn:
            total = 0
            while data := conn.recv(65536):      # b"" means the client closed
                print(f"  recv returned {len(data)} bytes", flush=True)
                total += len(data)
                conn.sendall(data)               # sendall = send_all in C
            print(f"client closed after {total} bytes", flush=True)

In [ ]:
%%writefile lab3/python/echo_client.py
"""echo_client.py - send lines from standard input, print the echo. Usage: python echo_client.py HOST PORT"""
import socket, sys
def recv_all(sock, n):
    """Read exactly n bytes, or raise if the peer closes first."""
    chunks = []
    while n > 0:
        data = sock.recv(n)
        if not data:
            raise ConnectionError("server closed the connection")
        chunks.append(data)
        n -= len(data)
    return b"".join(chunks)
with socket.create_connection((sys.argv[1], int(sys.argv[2]))) as s:
    for line in sys.stdin:
        data = line.encode()
        s.sendall(data)
        print(recv_all(s, len(data)).decode(), end="")

In [ ]:
P = Bg(f'{sys.executable} python/echo_server.py 5000')
sh(f'cd {LAB} && printf "one\\ntwo\\nthree\\n" | {sys.executable} python/echo_client.py 127.0.0.1 5000')
sh(f'cd {LAB} && ./blast 127.0.0.1 5000 300000')       # clientul C cu serverul Python
time.sleep(0.3); show('serverul Python', P.output()); P.stop()

---
# Partea D · Laboratorul 3 (Windows) — Socket-uri I cu Winsock
Colab rulează Linux, dar poate **construi programe Windows**: MinGW-w64 este un compilator încrucișat care produce fișiere `.exe` reale. Mai jos, sursele Winsock din fișă (și porturile cerute în Sarcinile 6–7) sunt compilate cu `-Wall -Wextra`, iar voi puteți descărca fișierele `.exe` și le puteți rula pe PC-ul vostru Windows exact cum descrie fișa.

## Ce este diferit în Winsock

In [ ]:
DIFF = [('Inițializare', 'niciuna', 'întâi WSAStartup(MAKEWORD(2,2), &wsa); WSACleanup() la sfârșit'),
        ('Antete și bibliotecă', '<sys/socket.h> <netinet/in.h> <arpa/inet.h> <netdb.h>', '<winsock2.h> <ws2tcpip.h>; legare cu ws2_32'),
        ('Tipul socket-ului', 'int (un descriptor de fișier)', 'SOCKET (un handle fără semn)'),
        ('Valoarea de eșec', '-1', 'INVALID_SOCKET (socket, accept) sau SOCKET_ERROR'),
        ('Închidere', 'close(fd)', 'closesocket(s) — close nu merge pe socket-uri'),
        ('Codul de eroare', 'errno, perror', 'WSAGetLastError()'),
        ('Citire și scriere', 'read/write sau recv/send', 'doar recv/send; lungimile sunt int'),
        ('Broken pipe', 'semnalul SIGPIPE', 'niciun semnal; send eșuează cu 10053/10054')]
for what, lin, win in DIFF:
    print(f'{what:20s} | Linux: {lin:52s} | Windows: {win}')

print('\nNumere de eroare frecvente:')
for name, num, posix in [('WSAECONNREFUSED', 10061, 'ECONNREFUSED'), ('WSAEADDRINUSE', 10048, 'EADDRINUSE'),
                          ('WSAECONNRESET', 10054, 'ECONNRESET'), ('WSAECONNABORTED', 10053, 'ECONNABORTED'),
                          ('WSANOTINITIALISED', 10093, '(fără echivalent: WSAStartup nu a fost apelat)')]:
    print(f'   {num}  {name:18s} ↔ Linux {posix}')

### Sursele Winsock

In [ ]:
%%writefile lab3/win/netio_win.h
/* netio_win.h - helpers for TCP byte streams on Windows (Winsock 2) - Computer Networks, Lab 3 */
#ifndef NETIO_WIN_H
#define NETIO_WIN_H
#include <winsock2.h>
#include <ws2tcpip.h>
#include <stdint.h>

/* Write exactly len bytes. Returns 0 on success, -1 on error. */
static inline int send_all(SOCKET s, const void *buf, size_t len) {
    const char *p = buf;
    while (len > 0) {
        int n = send(s, p, (int)len, 0);
        if (n == SOCKET_ERROR) return -1;
        p += n; len -= (size_t)n;
    }
    return 0;
}

/* Read exactly len bytes. Returns 1 on success, 0 if the peer closed first, -1 on error. */
static inline int recv_all(SOCKET s, void *buf, size_t len) {
    char *p = buf;
    while (len > 0) {
        int n = recv(s, p, (int)len, 0);
        if (n == SOCKET_ERROR) return -1;
        if (n == 0) return 0;
        p += n; len -= (size_t)n;
    }
    return 1;
}

static inline int send_msg(SOCKET s, const void *buf, uint32_t len) {
    uint32_t netlen = htonl(len);
    if (send_all(s, &netlen, sizeof netlen) < 0) return -1;
    return send_all(s, buf, len);
}

static inline long recv_msg(SOCKET s, void *buf, uint32_t cap) {
    uint32_t netlen;
    int r = recv_all(s, &netlen, sizeof netlen);
    if (r <= 0) return r;
    uint32_t len = ntohl(netlen);
    if (len > cap) return -1;
    r = recv_all(s, buf, len);
    return r <= 0 ? -1 : (long)len;
}
#endif

In [ ]:
%%writefile lab3/win/echo_client_win.c
/* echo_client_win.c - Winsock version of echo_client.c.
   Build:  gcc echo_client_win.c -o echo_client.exe -lws2_32     (MinGW-w64)
      or:  cl echo_client_win.c ws2_32.lib                         (Visual Studio)
   Usage:  echo_client.exe HOST PORT */
#include <stdio.h>
#include <string.h>
#include "netio_win.h"
#ifdef _MSC_VER
#pragma comment(lib, "ws2_32.lib")
#endif

int main(int argc, char *argv[]) {
    if (argc != 3) { fprintf(stderr, "usage: %s HOST PORT\n", argv[0]); return 1; }
    WSADATA wsa;
    if (WSAStartup(MAKEWORD(2, 2), &wsa) != 0) { fprintf(stderr, "WSAStartup failed\n"); return 1; }
    struct addrinfo hints = {0}, *res, *ai;
    hints.ai_family = AF_UNSPEC; hints.ai_socktype = SOCK_STREAM;
    int err = getaddrinfo(argv[1], argv[2], &hints, &res);
    if (err) { fprintf(stderr, "getaddrinfo failed: %d\n", err); WSACleanup(); return 1; }
    SOCKET s = INVALID_SOCKET;
    for (ai = res; ai != NULL; ai = ai->ai_next) {
        s = socket(ai->ai_family, ai->ai_socktype, ai->ai_protocol);
        if (s == INVALID_SOCKET) continue;
        if (connect(s, ai->ai_addr, (int)ai->ai_addrlen) == 0) break;
        closesocket(s); s = INVALID_SOCKET;
    }
    freeaddrinfo(res);
    if (s == INVALID_SOCKET) { fprintf(stderr, "connect failed: error %d\n", WSAGetLastError()); WSACleanup(); return 1; }
    char line[1024], reply[1024];
    while (fgets(line, sizeof line, stdin) != NULL) {
        size_t len = strlen(line);
        if (send_all(s, line, len) < 0) { fprintf(stderr, "send failed: %d\n", WSAGetLastError()); break; }
        int r = recv_all(s, reply, len);
        if (r <= 0) { fprintf(stderr, r == 0 ? "server closed the connection\n" : "recv failed\n"); break; }
        fwrite(reply, 1, len, stdout);
    }
    closesocket(s);
    WSACleanup();
    return 0;
}

In [ ]:
%%writefile lab3/win/echo_server_win.c
/* echo_server_win.c - Winsock version of echo_server.c (iterative, one client at a time).
   Build:  gcc echo_server_win.c -o echo_server.exe -lws2_32
   Usage:  echo_server.exe PORT */
#include <stdio.h>
#include <stdlib.h>
#include "netio_win.h"
#ifdef _MSC_VER
#pragma comment(lib, "ws2_32.lib")
#endif

int main(int argc, char *argv[]) {
    if (argc != 2) { fprintf(stderr, "usage: %s PORT\n", argv[0]); return 1; }
    setvbuf(stdout, NULL, _IONBF, 0);
    WSADATA wsa;
    if (WSAStartup(MAKEWORD(2, 2), &wsa) != 0) { fprintf(stderr, "WSAStartup failed\n"); return 1; }
    SOCKET ls = socket(AF_INET, SOCK_STREAM, IPPROTO_TCP);
    if (ls == INVALID_SOCKET) { fprintf(stderr, "socket: %d\n", WSAGetLastError()); return 1; }
    struct sockaddr_in addr = {0};
    addr.sin_family = AF_INET;
    /* Loopback only: avoids the Windows Firewall prompt. Use INADDR_ANY to accept other machines. */
    addr.sin_addr.s_addr = htonl(INADDR_LOOPBACK);
    addr.sin_port = htons((u_short)atoi(argv[1]));
    if (bind(ls, (struct sockaddr *)&addr, sizeof addr) == SOCKET_ERROR) { fprintf(stderr, "bind: %d\n", WSAGetLastError()); return 1; }
    if (listen(ls, 8) == SOCKET_ERROR) { fprintf(stderr, "listen: %d\n", WSAGetLastError()); return 1; }
    printf("listening on 127.0.0.1:%s\n", argv[1]);
    for (;;) {
        struct sockaddr_in peer; int plen = sizeof peer;
        SOCKET cs = accept(ls, (struct sockaddr *)&peer, &plen);
        if (cs == INVALID_SOCKET) { fprintf(stderr, "accept: %d\n", WSAGetLastError()); continue; }
        char ip[INET_ADDRSTRLEN];
        inet_ntop(AF_INET, &peer.sin_addr, ip, sizeof ip);
        printf("connection from %s:%u\n", ip, ntohs(peer.sin_port));
        char buf[65536]; int n; long total = 0;
        while ((n = recv(cs, buf, sizeof buf, 0)) > 0) {
            printf("  recv returned %d bytes\n", n);
            total += n;
            if (send_all(cs, buf, (size_t)n) < 0) break;
        }
        printf("client closed after %ld bytes\n", total);
        closesocket(cs);
    }
}

In [ ]:
%%writefile lab3/win/upper_server_win.c
/* upper_server_win.c - Winsock port of upper_server.c (Windows sheet, Task 6).
   Build:  gcc upper_server_win.c -o upper_server.exe -lws2_32
   Usage:  upper_server.exe PORT */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>
#include "netio_win.h"
#ifdef _MSC_VER
#pragma comment(lib, "ws2_32.lib")
#endif

int main(int argc, char *argv[]) {
    if (argc != 2) { fprintf(stderr, "usage: %s PORT\n", argv[0]); return 1; }
    setvbuf(stdout, NULL, _IONBF, 0);
    WSADATA wsa;
    if (WSAStartup(MAKEWORD(2, 2), &wsa) != 0) { fprintf(stderr, "WSAStartup failed\n"); return 1; }
    SOCKET ls = socket(AF_INET, SOCK_STREAM, IPPROTO_TCP);                 /* SOCKET, not int */
    if (ls == INVALID_SOCKET) { fprintf(stderr, "socket: %d\n", WSAGetLastError()); return 1; }
    struct sockaddr_in addr = {0};
    addr.sin_family = AF_INET;
    addr.sin_addr.s_addr = htonl(INADDR_LOOPBACK);
    addr.sin_port = htons((u_short)atoi(argv[1]));
    if (bind(ls, (struct sockaddr *)&addr, sizeof addr) == SOCKET_ERROR) { fprintf(stderr, "bind: %d\n", WSAGetLastError()); return 1; }
    if (listen(ls, 8) == SOCKET_ERROR) { fprintf(stderr, "listen: %d\n", WSAGetLastError()); return 1; }
    printf("listening on 127.0.0.1:%s\n", argv[1]);
    for (;;) {
        SOCKET cs = accept(ls, NULL, NULL);
        if (cs == INVALID_SOCKET) continue;
        char buf[4096]; size_t used = 0; int n;
        while ((n = recv(cs, buf + used, (int)(sizeof buf - used), 0)) > 0) {   /* lengths are int */
            used += (size_t)n;
            char *nl;
            while ((nl = memchr(buf, '\n', used)) != NULL) {
                size_t linelen = (size_t)(nl - buf) + 1;
                for (size_t i = 0; i < linelen; i++) buf[i] = (char)toupper((unsigned char)buf[i]);
                if (send_all(cs, buf, linelen) < 0) break;
                memmove(buf, buf + linelen, used - linelen);
                used -= linelen;
            }
            if (used == sizeof buf) { const char *e = "ERR line too long\n"; send_all(cs, e, strlen(e)); break; }
        }
        closesocket(cs);                                                    /* closesocket, not close */
    }
}

In [ ]:
%%writefile lab3/win/msg_server_win.c
/* msg_server_win.c - Winsock port of msg_server.c (Windows sheet, Task 7): replies "OK <n>".
   Build:  gcc msg_server_win.c -o msg_server.exe -lws2_32
   Usage:  msg_server.exe PORT */
#include <stdio.h>
#include <stdlib.h>
#include "netio_win.h"
#ifdef _MSC_VER
#pragma comment(lib, "ws2_32.lib")
#endif
#define CAP (1u << 20)

int main(int argc, char *argv[]) {
    if (argc != 2) { fprintf(stderr, "usage: %s PORT\n", argv[0]); return 1; }
    setvbuf(stdout, NULL, _IONBF, 0);
    WSADATA wsa;
    if (WSAStartup(MAKEWORD(2, 2), &wsa) != 0) { fprintf(stderr, "WSAStartup failed\n"); return 1; }
    SOCKET ls = socket(AF_INET, SOCK_STREAM, IPPROTO_TCP);
    if (ls == INVALID_SOCKET) { fprintf(stderr, "socket: %d\n", WSAGetLastError()); return 1; }
    struct sockaddr_in addr = {0};
    addr.sin_family = AF_INET;
    addr.sin_addr.s_addr = htonl(INADDR_LOOPBACK);
    addr.sin_port = htons((u_short)atoi(argv[1]));
    if (bind(ls, (struct sockaddr *)&addr, sizeof addr) == SOCKET_ERROR) { fprintf(stderr, "bind: %d\n", WSAGetLastError()); return 1; }
    if (listen(ls, 8) == SOCKET_ERROR) { fprintf(stderr, "listen: %d\n", WSAGetLastError()); return 1; }
    printf("listening on 127.0.0.1:%s\n", argv[1]);
    char *buf = malloc(CAP);
    if (!buf) return 1;
    for (;;) {
        SOCKET cs = accept(ls, NULL, NULL);
        if (cs == INVALID_SOCKET) continue;
        long len;
        while ((len = recv_msg(cs, buf, CAP)) > 0) {
            printf("message of %ld bytes\n", len);
            char reply[32];
            int rl = snprintf(reply, sizeof reply, "OK %ld", len);
            if (send_msg(cs, reply, (uint32_t)rl) < 0) break;
        }
        printf(len < 0 ? "bad or too long message: closing\n" : "client closed\n");
        closesocket(cs);
    }
}

In [ ]:
%%writefile lab3/win/msg_client_win.c
/* msg_client_win.c - Winsock port of msg_client.c: sends every argument before reading replies.
   Build:  gcc msg_client_win.c -o msg_client.exe -lws2_32
   Usage:  msg_client.exe HOST PORT MSG... */
#include <stdio.h>
#include <string.h>
#include "netio_win.h"
#ifdef _MSC_VER
#pragma comment(lib, "ws2_32.lib")
#endif

int main(int argc, char *argv[]) {
    if (argc < 4) { fprintf(stderr, "usage: %s HOST PORT MSG...\n", argv[0]); return 1; }
    WSADATA wsa;
    if (WSAStartup(MAKEWORD(2, 2), &wsa) != 0) { fprintf(stderr, "WSAStartup failed\n"); return 1; }
    struct addrinfo hints = {0}, *res, *ai;
    hints.ai_family = AF_UNSPEC; hints.ai_socktype = SOCK_STREAM;
    if (getaddrinfo(argv[1], argv[2], &hints, &res)) { fprintf(stderr, "getaddrinfo failed\n"); return 1; }
    SOCKET s = INVALID_SOCKET;
    for (ai = res; ai != NULL; ai = ai->ai_next) {
        s = socket(ai->ai_family, ai->ai_socktype, ai->ai_protocol);
        if (s == INVALID_SOCKET) continue;
        if (connect(s, ai->ai_addr, (int)ai->ai_addrlen) == 0) break;
        closesocket(s); s = INVALID_SOCKET;
    }
    freeaddrinfo(res);
    if (s == INVALID_SOCKET) { fprintf(stderr, "connect failed: error %d\n", WSAGetLastError()); return 1; }
    for (int i = 3; i < argc; i++)
        if (send_msg(s, argv[i], (uint32_t)strlen(argv[i])) < 0) { fprintf(stderr, "send: %d\n", WSAGetLastError()); return 1; }
    for (int i = 3; i < argc; i++) {
        char reply[64];
        long n = recv_msg(s, reply, sizeof reply - 1);
        if (n <= 0) { fprintf(stderr, "no reply\n"); break; }
        reply[n] = '\0';
        printf("%s -> %s\n", argv[i], reply);
    }
    closesocket(s);
    WSACleanup();
    return 0;
}

In [ ]:
WIN = os.path.join(LAB, 'win')
for prog in ['echo_client', 'echo_server', 'upper_server', 'msg_server', 'msg_client']:
    sh(f'cd {WIN} && x86_64-w64-mingw32-gcc -Wall -Wextra -O2 {prog}_win.c -o {prog}.exe -lws2_32 && echo "{prog}.exe construit, fără avertismente"')
sh(f'cd {WIN} && file *.exe')

In [ ]:
# Cea mai frecventă eroare de compilare din fișa Windows: uitarea lui -lws2_32
sh(f'cd {WIN} && x86_64-w64-mingw32-gcc echo_server_win.c -o /tmp/broken.exe 2>&1 | grep -m3 "undefined reference"')

### Descărcați și rulați pe Windows
Celula de mai jos împachetează sursele și fișierele `.exe` în `lab3_windows.zip`. Pe PC: dezarhivați, deschideți PowerShell în dosar și rulați `.\echo_server.exe 5000` într-o fereastră și `.\echo_client.exe 127.0.0.1 5000` în alta (Sarcinile 2–3 din fișa Windows).

In [ ]:
import zipfile
with zipfile.ZipFile('lab3_windows.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(WIN)):
        z.write(os.path.join(WIN, f), f'lab3_windows/{f}')
    for f in ('echo_server.py', 'echo_client.py'):
        z.write(os.path.join(LAB, 'python', f), f'lab3_windows/python/{f}')
print('lab3_windows.zip:', os.path.getsize('lab3_windows.zip'), 'octeți')
if IN_COLAB:
    from google.colab import files
    files.download('lab3_windows.zip')

### Opțional: rulați aici fișierele `.exe` cu Wine
Wine rulează programe Windows pe Linux și implementează Winsock peste socket-urile Linux. Setați `USE_WINE = True` (instalarea durează 2–4 minute). Un client Windows poate vorbi atunci cu un server Linux și invers — protocolul este același, doar API-ul diferă.

In [ ]:
USE_WINE = False

if USE_WINE:
    if not (have('wine') or have('wine64')):
        sh(f'DEBIAN_FRONTEND=noninteractive {SUDO}apt-get -qq install -y wine wine64 > /dev/null 2>&1; echo installed', timeout=900)
    WINE = 'wine' if have('wine') else 'wine64'
    os.environ.update(WINEDEBUG='-all', WINEPREFIX='/tmp/wineprefix')
    WS = Bg(f'{WINE} ./echo_server.exe 5100', cwd=WIN, settle=40)        # prima pornire a lui Wine este lentă
    sh(f'cd {WIN} && printf "salut de la un client Windows\\n" | timeout 60 {WINE} ./echo_client.exe 127.0.0.1 5100', timeout=90)
    sh(f'cd {LAB} && ./blast 127.0.0.1 5100 300000')                    # client Linux → server Windows
    time.sleep(1); show('echo_server.exe (sub Wine)', WS.output()); WS.stop()
else:
    print('Wine nu este folosit (setați USE_WINE = True pentru a încerca).')

### Întrebări care depind de Windows însuși
* **Q4 (Windows)** — `connect` la un port închis eșuează cu **10061** după circa **2 s** în Windows: când SYN-ului i se răspunde cu RST, Windows mai reîncearcă SYN-ul de două ori înainte să renunțe; Linux raportează `ECONNREFUSED` la primul RST.
* **Q7 (Windows)** — fără `htons`, portul 5000 devine **34835**, ca în Linux: ambele rulează pe procesoare x86 little-endian (vezi calculul de mai jos).
* **Q16 (Windows)** — în Windows, `SO_REUSEADDR` permite unui *al doilea* program să lege un port deja folosit (chiar să-i fure traficul). `SO_EXCLUSIVEADDRUSE` interzice oricărui alt socket să lege aceeași adresă și același port; serverele ar trebui să îl seteze.

In [ ]:
print('Q7: 5000 fără htons →', struct.unpack('!H', struct.pack('<H', 5000))[0])

### Sarcina 5 (Windows) din PowerShell — opțional
Fragmentul PowerShell din fișă folosește `TcpClient` din .NET, care există și în PowerShell 7 pe Linux. Setați `INSTALL_PWSH = True` pentru a-l instala și a trimite 300 000 de octeți cu un singur `Write` către serverul nostru echo în C.

In [ ]:
INSTALL_PWSH = False

if INSTALL_PWSH and not have('pwsh'):
    ver = re.search(r'VERSION_ID="([\d.]+)"', open('/etc/os-release').read()).group(1)
    sh(f'wget -q https://packages.microsoft.com/config/ubuntu/{ver}/packages-microsoft-prod.deb -O /tmp/ms.deb && '
       f'{SUDO}dpkg -i /tmp/ms.deb > /dev/null && {SUDO}apt-get -qq update > /dev/null && '
       f'{SUDO}apt-get -qq install -y powershell > /dev/null; pwsh -Version', timeout=600)
PS = r"""
$c = New-Object System.Net.Sockets.TcpClient('127.0.0.1', 5000)
$s = $c.GetStream()
$data = [byte[]]::new(300000)
$s.Write($data, 0, $data.Length)          # o singură scriere de 300 000 de octeți
$c.Client.Shutdown('Send')                # ca shutdown(SHUT_WR)
$buf = [byte[]]::new(65536); while ($s.Read($buf, 0, $buf.Length) -gt 0) { }
$c.Close()
"""
if have('pwsh'):
    S = Bg('./echo_server 5000')
    open('/tmp/blast.ps1', 'w').write(PS); sh('pwsh -NoProfile -File /tmp/blast.ps1', timeout=60)
    time.sleep(0.5); show('echo_server', S.output()); S.stop()
else:
    print('PowerShell nu este instalat (setați INSTALL_PWSH = True pentru a încerca).')

**✍️ Răspunsurile tale** (dublu-clic pentru a edita)

- **Q3 (Windows) — diferențele Winsock din cod; care dintre ele produce eșec imediat dacă e uitată?:** 
- **Q11–Q12 (Windows) — încadrarea cu linii în Windows:** 
- **Q13–Q14 (Windows) — prefix de lungime; verificarea dimensiunii fișierului cu (Get-Item file).Length:** 

---
### Săptămâna viitoare
Cursul 4 continuă cu Web-ul: cookie-uri, cache, HTTP/2, HTTP/3 și rețele de distribuție a conținutului. Laboratorul 4 trece la socket-uri UDP și ordinea octeților.